# 01 — Data preparation & EDA
Run the kernel from the project root (cwd must contain `config.json`). Flow: splits via `prepare.py` → manifest summary → EDA figures saved to `processed/eda/`.

In [ ]:
import json
import os

assert os.path.exists("config.json"), f"cwd is {os.getcwd()}: start the kernel in the project root"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.stats import fisher_exact

import prepare
from data import build_cache, build_transforms, load_config, make_weighted_sampler, splits_hash
from plots import _save_show as save_show

cfg = load_config("config.json")
classes = cfg["classes"]
n_cls = len(classes)
names = [cfg["class_names"][c] for c in classes]
short = [n[:12] for n in names]
PAL = [plt.get_cmap("tab10")(i) for i in range(n_cls)]
SPLITS = ("train", "val", "test")
EDA_DIR = os.path.join(os.path.dirname(cfg["paths"]["train_csv"]) or ".", "eda")
os.makedirs(EDA_DIR, exist_ok=True)

## 1. Prepare splits
`FORCE_PREPARE = False` reuses an existing manifest. Set it to `True` after editing `prep_config.json`: this rewrites the CSVs and deletes the stale `cache_*` files.

First run after the `prepare.py` change: `FORCE_PREPARE = True` (set below). When it finishes, restart the kernel (stale memmaps, `prepare` already imported), set `FORCE_PREPARE = False` and rerun from the top. Delete the old `runs/<config_hash>/` contents: their `splits_hash` no longer matches.

In [ ]:
FORCE_PREPARE = True

if FORCE_PREPARE or not os.path.exists(prepare.manifest_path(cfg)):
    manifest = prepare.run("config.json", "prep_config.json")
else:
    with open(prepare.manifest_path(cfg)) as f:
        manifest = json.load(f)
    print("prepare skipped: manifest found")
print("splits_hash:", splits_hash(cfg))

In [ ]:
drop_on = manifest["prep_config"]["drop_train_outliers"]
exp = sum(manifest["train_flag_by_class"]["any_drop_flag"].values()) if drop_on else 0
assert manifest["removal_log"]["train_outlier_flag"]["n"] == exp
assert manifest["rows"]["total"] == manifest["rows"]["after_cleaning"] - exp
print(f"drop_train_outliers={drop_on} | train rows dropped={exp} | train={manifest['rows']['train']}")

## 2. Manifest summary
Removal log = rows dropped before the split (data-independent issues) plus `train_outlier_flag` (train rows dropped for bright/contrast flags; 0 when `drop_train_outliers` is false).

Flag thresholds are train-only IQR bounds; flag counts are taken before any optional train drop. `age_outlier` is informational: it is reported in the flag tables but excluded from the drop (only `bright_outlier` and `contrast_outlier` can drop train rows).

In [ ]:
print(f"rows: {manifest['rows']}")
print(f"ratios: {manifest['ratios']} | n_splits: {manifest['n_splits']} | sklearn: {manifest['sklearn_version']}")
cc = pd.DataFrame(manifest["class_counts"]).T[classes]
cc["total"] = cc.sum(axis=1)
display(cc)
display(pd.DataFrame({r: {**v["by_class"], "total": v["n"]} for r, v in manifest["removal_log"].items()}).T)
display(pd.DataFrame(manifest["flag_thresholds"], index=["lower", "upper"]).T)
display(pd.DataFrame(manifest["flag_counts"]).T)

## 3. EDA figures
Figures describe the cleaned dataset (all three splits unless stated); they are descriptive and no decision depends on them. Metadata columns (`age`, `sex`, `localization`) come from `eda_columns` in `prep_config.json`; the models are image-only.

In [ ]:
dfs = {s: pd.read_csv(cfg["paths"][f"{s}_csv"]) for s in SPLITS}
all_df = pd.concat([d.assign(split=s) for s, d in dfs.items()], ignore_index=True)
x = np.arange(n_cls)
print({s: len(d) for s, d in dfs.items()}, "| age missing:", int(all_df["age"].isna().sum()))

**Class distribution.** `nv` dominates and `df`/`vasc` are rare; the weighted sampler, focal loss and logit adjustment in the pipeline address this, not the data files.

In [ ]:
cnt = np.bincount(all_df["label_idx"].to_numpy(np.int64), minlength=n_cls)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, cnt, color=PAL)
for i, c in enumerate(cnt):
    ax.text(i, c + 0.01 * cnt.max(), f"{c}\n({c / cnt.sum() * 100:.1f}%)", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right")
ax.set_ylabel("Images")
ax.set_title("Class distribution (cleaned dataset)", fontweight="bold")
ax.grid(axis="y", alpha=0.35)
fig.tight_layout()
save_show(fig, os.path.join(EDA_DIR, "class_distribution.png"))

**Age, sex, localization.** Missing ages are left as NaN (no imputation); boxplots use the available values only.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
age = [all_df.loc[(all_df["label_idx"] == i) & all_df["age"].notna(), "age"].to_numpy() for i in range(n_cls)]
axes[0].boxplot(age)
axes[0].set_xticks(range(1, n_cls + 1))
axes[0].set_xticklabels(short, rotation=40, ha="right")
axes[0].set_title("Age by class")
axes[0].set_ylabel("Age")
sex = all_df.groupby(["label_idx", "sex"]).size().unstack(fill_value=0).reindex(range(n_cls), fill_value=0)
sex.index = short
sex.plot(kind="bar", ax=axes[1], colormap="Set2", width=0.8, rot=40)
axes[1].set_title("Sex by class")
axes[1].set_ylabel("Images")
axes[1].legend(title="Sex", fontsize=8)
loc = all_df["localization"].value_counts().head(12)
axes[2].barh(loc.index.tolist(), loc.values, color=[plt.get_cmap("viridis")(v) for v in np.linspace(0, 1, len(loc))])
axes[2].invert_yaxis()
axes[2].set_title("Top 12 localizations")
axes[2].set_xlabel("Images")
for a in axes:
    a.grid(alpha=0.35)
fig.tight_layout()
save_show(fig, os.path.join(EDA_DIR, "age_sex_localization.png"))

**Missing metadata.** `age` missing = NaN; `sex` and `localization` missing = the string "unknown" (not detected by `isna()`). Counts and % over all splits, then per split, per class and per class x split.

In [ ]:
flag_cols = ["age_nan", "sex_unknown", "loc_unknown"]
ind = pd.DataFrame({
    "age_nan": all_df["age"].isna(),
    "sex_unknown": all_df["sex"].astype(str).str.strip().str.lower().eq("unknown"),
    "loc_unknown": all_df["localization"].astype(str).str.strip().str.lower().eq("unknown"),
})
ind["class"] = pd.Categorical(all_df["label_idx"].map(dict(enumerate(classes))), classes, ordered=True)
ind["split"] = pd.Categorical(all_df["split"], list(SPLITS), ordered=True)
ind["all"] = "all"


def miss_summary(keys) -> pd.DataFrame:
    g = ind.groupby(keys, observed=True)[flag_cols]
    n, pct = g.sum().astype(int), g.mean().mul(100).round(2)
    out = pd.DataFrame({"rows": g.size()})
    for k in flag_cols:
        out[f"{k}_n"] = n[k]
        out[f"{k}_%"] = pct[k]
    return out


for title, keys in (("all splits", "all"), ("per split", "split"), ("per class", "class"), ("per class x split", ["class", "split"])):
    print(title)
    display(miss_summary(keys))

**Missingness vs class (train only).** For each indicator (age NaN, sex unknown, localization unknown): 7 one-vs-rest 2x2 Fisher exact tests (rows: class c / others; columns: indicator yes / no). Holm correction is applied within each indicator (family = 7 tests). Odds ratio = a*d / (b*c); 0 or inf when a cell is empty.

In [ ]:
tr = dfs["train"]
lab = tr["label_idx"].to_numpy(np.int64)
IND = {
    "age_nan": tr["age"].isna().to_numpy(),
    "sex_unknown": tr["sex"].astype(str).str.strip().str.lower().eq("unknown").to_numpy(),
    "loc_unknown": tr["localization"].astype(str).str.strip().str.lower().eq("unknown").to_numpy(),
}


def holm(p: np.ndarray) -> np.ndarray:
    m = len(p)
    adj = np.empty(m)
    run = 0.0
    for rank, i in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(run, 1.0)
    return adj


rows = []
for name, flag in IND.items():
    if flag.sum() == 0 or flag.all():
        print(f"{name}: no positives (or all positive) in train, skipped")
        continue
    block = []
    for k in range(n_cls):
        in_k = lab == k
        a, b = int((flag & in_k).sum()), int((~flag & in_k).sum())
        c_, d = int((flag & ~in_k).sum()), int((~flag & ~in_k).sum())
        r = fisher_exact([[a, b], [c_, d]])
        block.append({
            "indicator": name, "class": classes[k],
            "yes_in_class": a, "no_in_class": b, "yes_other": c_, "no_other": d,
            "odds_ratio": float(r[0]), "p_raw": float(r[1]),
        })
    adj = holm(np.array([row["p_raw"] for row in block]))
    for row, pa in zip(block, adj):
        row["p_holm"] = float(pa)
    rows.extend(block)

fisher_df = pd.DataFrame(rows)
display(fisher_df.style.format({"odds_ratio": "{:.2f}", "p_raw": "{:.2e}", "p_holm": "{:.2e}"}))

**Decision rule: descriptive only.** No row is removed and nothing is imputed, because no model uses metadata. Significant rows document how missingness correlates with class; they do not trigger any action in the pipeline.

**Split class distributions.** Percentages should be close across splits (stratified by image-level label, grouped by `lesion_id`); with few `df`/`vasc` images per split, per-class metrics on val/test stay noisy.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, s in zip(axes, SPLITS):
    d = dfs[s]
    c = np.bincount(d["label_idx"].to_numpy(np.int64), minlength=n_cls)
    ax.bar(x, c, color=PAL)
    for i, v in enumerate(c):
        ax.text(i, v + 0.01 * c.max(), f"{v / c.sum() * 100:.0f}%", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(short, rotation=45, ha="right")
    ax.set_title(f"{s} (images={len(d)}, lesions={d['lesion_id'].nunique()})", fontweight="bold")
    ax.set_ylabel("Images")
    ax.grid(axis="y", alpha=0.35)
fig.suptitle("Class distribution across splits", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
save_show(fig, os.path.join(EDA_DIR, "split_class_distribution.png"))

**Effective class distribution.** One epoch of `make_weighted_sampler` on train labels (indices only, no images loaded) vs the natural train counts. `uniform_n` = N / n_classes, the expected count per class under balanced sampling. The draw is a single seeded sample, so small deviations from `uniform_n` are noise.

In [ ]:
tr = dfs["train"]
lab = tr["label_idx"].to_numpy(np.int64)
torch.manual_seed(cfg["seed"])
drawn = np.asarray(list(make_weighted_sampler(lab, n_cls)), dtype=np.int64)
base = np.bincount(lab, minlength=n_cls)
samp = np.bincount(lab[drawn], minlength=n_cls)
tab = pd.DataFrame({
    "train_n": base,
    "train_%": base / base.sum() * 100,
    "sampled_n": samp,
    "sampled_%": samp / samp.sum() * 100,
    "uniform_n": np.full(n_cls, base.sum() / n_cls),
}, index=classes).round(2)
tab.loc["total"] = [base.sum(), 100.0, samp.sum(), 100.0, base.sum()]
display(tab)

**Brightness vs contrast.** Dashed lines are the IQR bounds fitted on train only (`manifest["flag_thresholds"]`) and applied unchanged to val/test. If `drop_train_outliers` is true, flagged train rows are already absent from the left panel.

In [ ]:
(b_lo, b_hi), (c_lo, c_hi) = manifest["flag_thresholds"]["brightness"], manifest["flag_thresholds"]["contrast"]


def flagged(d: pd.DataFrame) -> pd.Series:
    return (d["bright_outlier"] | d["contrast_outlier"]).astype(bool)


fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
tr = dfs["train"]
f = flagged(tr)
axes[0].scatter(tr.loc[~f, "brightness"], tr.loc[~f, "contrast"], s=5, alpha=0.3, color="steelblue", label=f"normal ({int((~f).sum())})")
axes[0].scatter(tr.loc[f, "brightness"], tr.loc[f, "contrast"], s=20, color="red", zorder=5, label=f"flagged ({int(f.sum())})")
dr = pd.DataFrame(manifest["removal_log"]["train_outlier_flag"].get("rows", []), columns=["brightness", "contrast"])
if len(dr):
    axes[0].scatter(dr["brightness"], dr["contrast"], s=25, marker="x", color="red", zorder=5, label=f"dropped ({len(dr)})")
axes[0].set_title("train", fontweight="bold")
for s, col in (("val", "darkorange"), ("test", "seagreen")):
    d = dfs[s]
    f = flagged(d)
    axes[1].scatter(d["brightness"], d["contrast"], s=8, alpha=0.4, color=col, label=f"{s} ({len(d)}, flagged {int(f.sum())})")
    axes[1].scatter(d.loc[f, "brightness"], d.loc[f, "contrast"], s=25, marker="x", color="red", zorder=5)
axes[1].set_title("val / test (flagged = red x)", fontweight="bold")
for ax in axes:
    for v in (b_lo, b_hi):
        ax.axvline(v, color="red", ls="--", lw=0.9)
    for v in (c_lo, c_hi):
        ax.axhline(v, color="red", ls="--", lw=0.9)
    ax.set_xlabel("Mean brightness")
    ax.set_ylabel("Contrast (std)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.35)
bx = pd.concat([all_df["brightness"], dr["brightness"]])
cy = pd.concat([all_df["contrast"], dr["contrast"]])
axes[0].set_xlim(bx.min() - 0.03 * (bx.max() - bx.min()), bx.max() + 0.03 * (bx.max() - bx.min()))
axes[0].set_ylim(cy.min() - 0.03 * (cy.max() - cy.min()), cy.max() + 0.03 * (cy.max() - cy.min()))
fig.suptitle(f"Brightness vs contrast, thresholds from train only (IQR x {manifest['prep_config']['iqr_factor']})", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
save_show(fig, os.path.join(EDA_DIR, "brightness_contrast.png"))

In [ ]:
(b_lo, b_hi), (c_lo, c_hi) = manifest["flag_thresholds"]["brightness"], manifest["flag_thresholds"]["contrast"]
dr = pd.DataFrame(
    manifest["removal_log"]["train_outlier_flag"].get("rows", []),
    columns=["image_id", "dx", "filepath", "brightness", "contrast"],
)
cmap = {c: PAL[i] for i, c in enumerate(classes)}
order = ["nv"] + [c for c in classes if c != "nv"]  # nv drawn first, so it sits underneath


def flagged(d: pd.DataFrame) -> pd.Series:
    return (d["bright_outlier"] | d["contrast_outlier"]).astype(bool)


fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharex=True, sharey=True)
tr = dfs["train"]
f = flagged(tr)
for c in order:
    k = tr[(tr["dx"] == c) & ~f]
    axes[0].scatter(k["brightness"], k["contrast"], s=6, alpha=0.35, color=cmap[c], label=f"{c} ({len(k)})")
    g = dr[dr["dx"] == c]
    axes[0].scatter(g["brightness"], g["contrast"], s=28, marker="x", color=cmap[c], linewidths=1.2, zorder=5)
    h = tr[(tr["dx"] == c) & f]
    axes[0].scatter(h["brightness"], h["contrast"], s=28, marker="x", color=cmap[c], linewidths=1.2, zorder=5)
axes[0].set_title(f"train (kept {int((~f).sum())}, dropped {len(dr)}, flagged-kept {int(f.sum())})", fontweight="bold")
for s, mk in (("val", "o"), ("test", "^")):
    d = dfs[s]
    fl = flagged(d)
    for c in order:
        m = d["dx"] == c
        axes[1].scatter(d.loc[m & ~fl, "brightness"], d.loc[m & ~fl, "contrast"], s=8 if s == "val" else 14,
                        alpha=0.4, marker=mk, color=cmap[c])
        axes[1].scatter(d.loc[m & fl, "brightness"], d.loc[m & fl, "contrast"], s=28, marker="x",
                        color=cmap[c], linewidths=1.2, zorder=5)
axes[1].set_title(
    f"val (o, {len(dfs['val'])}, flagged {int(flagged(dfs['val']).sum())}) / "
    f"test (^, {len(dfs['test'])}, flagged {int(flagged(dfs['test']).sum())}); x = flagged",
    fontweight="bold", fontsize=10,
)
for ax in axes:
    for v in (b_lo, b_hi):
        ax.axvline(v, color="red", ls="--", lw=0.9)
    for v in (c_lo, c_hi):
        ax.axhline(v, color="red", ls="--", lw=0.9)
    ax.set_xlabel("Mean brightness")
    ax.set_ylabel("Contrast (std)")
    ax.grid(alpha=0.35)
axes[0].legend(fontsize=8, markerscale=2, title="class (kept)")
bx = pd.concat([all_df["brightness"], dr["brightness"]])
cy = pd.concat([all_df["contrast"], dr["contrast"]])
axes[0].set_xlim(bx.min() - 0.03 * (bx.max() - bx.min()), bx.max() + 0.03 * (bx.max() - bx.min()))
axes[0].set_ylim(cy.min() - 0.03 * (cy.max() - cy.min()), cy.max() + 0.03 * (cy.max() - cy.min()))
fig.suptitle(f"Brightness vs contrast by class, thresholds from train only (IQR x {manifest['prep_config']['iqr_factor']})", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
save_show(fig, os.path.join(EDA_DIR, "brightness_contrast.png"))

**Outliers.** Train flags by class (before any optional drop) and removal log.

In [ ]:
fl = pd.DataFrame(manifest["train_flag_by_class"]).T[classes]
fl["total"] = fl.sum(axis=1)
display(fl)
display(pd.DataFrame({r: {**v["by_class"], "total": v["n"]} for r, v in manifest["removal_log"].items()}).T)

**Augmentation grid.** One train image per class, shown as the model sees it: cached 224x224 input, then `data.build_transforms(cfg, train=True)`. The transform is the same for every class; minority classes are augmented more only because the sampler draws them more often. Output varies on each run.

In [ ]:
N_AUG = 4
tf = build_transforms(cfg, train=True)
tr = dfs["train"]
lab = tr["label_idx"].to_numpy(np.int64)
mm = np.load(build_cache(cfg, "train", tr), mmap_mode="r")
rng = np.random.default_rng(cfg["seed"])
pick = [int(rng.choice(np.where(lab == c)[0])) for c in range(n_cls)]
mean, std = np.asarray(cfg["norm_mean"]), np.asarray(cfg["norm_std"])
fig, axes = plt.subplots(n_cls, N_AUG + 1, figsize=(2.4 * (N_AUG + 1), 2.4 * n_cls), squeeze=False)
for r, i in enumerate(pick):
    img = np.array(mm[i])
    axes[r, 0].imshow(img)
    axes[r, 0].set_ylabel(short[r], fontsize=8)
    for k in range(1, N_AUG + 1):
        t = tf(image=img)["image"].permute(1, 2, 0).numpy()
        axes[r, k].imshow((t * std + mean).clip(0, 1))
for r in range(n_cls):
    for k in range(N_AUG + 1):
        axes[r, k].set_xticks([])
        axes[r, k].set_yticks([])
for k in range(N_AUG + 1):
    axes[0, k].set_title("Original" if k == 0 else f"Aug {k}", fontsize=9)
fig.suptitle("Train augmentation (data.build_transforms, train=True)", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.97))
save_show(fig, os.path.join(EDA_DIR, "augmentations.png"))
del mm